In [0]:
df = (
    spark.read
        .option("multiLine", "false")
        .option("mode", "PERMISSIVE")
        .json("/Volumes/workspace/default/musical_instrument/cleaned_Review_Musical_Instruments.jsonl")
)

In [0]:
df.printSchema()

root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)



In [0]:
from pyspark.sql.functions import col

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.columns

Rows: 2607415
Columns: 9


['asin',
 'helpful_vote',
 'parent_asin',
 'rating',
 'text',
 'timestamp',
 'title',
 'user_id',
 'verified_purchase']

2. Missing Value Analysis

In [0]:
numeric_type = ['float', 'double']
num_cols = [c for c, t in df.dtypes if t in numeric_type]

from pyspark.sql.functions import isnan, when, count, col

missing_df = df.select([count(when(col(c).isNull() | (isnan(col(c)) if c in num_cols else False), c)).alias(c)
    for c in df.columns
])

missing_df.show(truncate=False)

+----+------------+-----------+------+----+---------+-----+-------+-----------------+
|asin|helpful_vote|parent_asin|rating|text|timestamp|title|user_id|verified_purchase|
+----+------------+-----------+------+----+---------+-----+-------+-----------------+
|0   |0           |0          |0     |0   |0        |0    |0      |0                |
+----+------------+-----------+------+----+---------+-----+-------+-----------------+



In [0]:
total_rows = df.count()

missing_pct = missing_df.select([
    (col(c) / total_rows * 100).alias(c)
    for c in missing_df.columns
])

missing_pct.show(truncate=False)

+----+------------+-----------+------+----+---------+-----+-------+-----------------+
|asin|helpful_vote|parent_asin|rating|text|timestamp|title|user_id|verified_purchase|
+----+------------+-----------+------+----+---------+-----+-------+-----------------+
|0.0 |0.0         |0.0        |0.0   |0.0 |0.0      |0.0  |0.0    |0.0              |
+----+------------+-----------+------+----+---------+-----+-------+-----------------+



Basic Preview (Sanity Check)

In [0]:
df.show(5, truncate=False)

+----------+------------+-----------+------+-----------------------------------------------------------------------------------------------------------------------------------------------------+-------------+------------------------------------------------+----------------------------+-----------------+
|asin      |helpful_vote|parent_asin|rating|text                                                                                                                                                 |timestamp    |title                                           |user_id                     |verified_purchase|
+----------+------------+-----------+------+-----------------------------------------------------------------------------------------------------------------------------------------------------+-------------+------------------------------------------------+----------------------------+-----------------+
|B003LPTAYI|0           |B003LPTAYI |5.0   |Great headphones, comfortable and sound i

In [0]:
df.describe(["rating", "helpful_vote"]).show()

+-------+------------------+------------------+
|summary|            rating|      helpful_vote|
+-------+------------------+------------------+
|  count|           2607415|           2607415|
|   mean| 4.252030842807915|0.9303072199860781|
| stddev|1.2938267982779246| 9.002608455616059|
|    min|               1.0|                -1|
|    max|               5.0|              4650|
+-------+------------------+------------------+



Rating Distribution

In [0]:
from pyspark.sql.functions import count

df.groupBy("rating") \
  .agg(count("*").alias("review_count")) \
  .orderBy("rating") \
  .show()

+------+------------+
|rating|review_count|
+------+------------+
|   1.0|      238196|
|   2.0|      113785|
|   3.0|      166095|
|   4.0|      323937|
|   5.0|     1765402|
+------+------------+



Verified vs Unverified Purchases

In [0]:
df.groupBy("verified_purchase") \
  .count() \
  .show()


+-----------------+-------+
|verified_purchase|  count|
+-----------------+-------+
|             true|2430873|
|            false| 176542|
+-----------------+-------+



Rating vs Verified Purchase

In [0]:
df.groupBy("verified_purchase", "rating") \
  .count() \
  .orderBy("verified_purchase", "rating") \
  .show()


+-----------------+------+-------+
|verified_purchase|rating|  count|
+-----------------+------+-------+
|            false|   1.0|  19773|
|            false|   2.0|   9408|
|            false|   3.0|  12919|
|            false|   4.0|  27158|
|            false|   5.0| 107284|
|             true|   1.0| 218423|
|             true|   2.0| 104377|
|             true|   3.0| 153176|
|             true|   4.0| 296779|
|             true|   5.0|1658118|
+-----------------+------+-------+



Distribution of helpful votes

In [0]:
df.select("helpful_vote").describe().show()


+-------+------------------+
|summary|      helpful_vote|
+-------+------------------+
|  count|           2607415|
|   mean|0.9303072199860781|
| stddev| 9.002608455616059|
|    min|                -1|
|    max|              4650|
+-------+------------------+



Reviews with helpful votes > 0

In [0]:
df.filter(df.helpful_vote > 0) \
  .groupBy("helpful_vote") \
  .count() \
  .orderBy("helpful_vote") \
  .show(10)


+------------+------+
|helpful_vote| count|
+------------+------+
|           1|341642|
|           2|118631|
|           3| 58121|
|           4| 33228|
|           5| 21342|
|           6| 14754|
|           7| 10641|
|           8|  8135|
|           9|  6154|
|          10|  4971|
+------------+------+
only showing top 10 rows


Review Length Analysis

In [0]:
from pyspark.sql.functions import length

df = df.withColumn("review_length", length("text"))


In [0]:
df.select("review_length").describe().show()


+-------+------------------+
|summary|     review_length|
+-------+------------------+
|  count|           2607415|
|   mean|214.50152162198958|
| stddev|355.19569091534254|
|    min|                 0|
|    max|             24586|
+-------+------------------+



Review Length vs Rating

In [0]:
df.groupBy("rating") \
  .avg("review_length") \
  .orderBy("rating") \
  .show()


+------+------------------+
|rating|avg(review_length)|
+------+------------------+
|   1.0| 232.5060790273556|
|   2.0|301.47644241332335|
|   3.0| 298.4447334356844|
|   4.0|287.95604392212067|
|   5.0| 185.0905204593628|
+------+------------------+



Convert timestamp to date

In [0]:
from pyspark.sql.functions import from_unixtime, to_date

df = df.withColumn(
    "review_date",
    to_date(from_unixtime(df.timestamp / 1000))
)


Reviews per year

In [0]:
from pyspark.sql.functions import year

df.groupBy(year("review_date").alias("year")) \
  .count() \
  .orderBy("year") \
  .show()


+----+------+
|year| count|
+----+------+
|2015|241910|
|2016|271023|
|2017|261260|
|2018|273353|
|2019|346548|
|2020|390665|
|2021|390310|
|2022|301080|
|2023|131266|
+----+------+



Top Products
Most Reviewed Products

In [0]:
df.groupBy("asin") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(10)


+----------+-----+
|      asin|count|
+----------+-----+
|B01H74YV56| 4959|
|B00DY1F2CS| 4945|
|B018FCZKR2| 3911|
|B004XNK7AI| 3778|
|B00O4L3F9E| 3708|
|B01A6B0ICC| 3489|
|B016C4ZG74| 3239|
|B008AOH1O6| 3167|
|B0006NDF8A| 3108|
|B06XCKGLTP| 3101|
+----------+-----+
only showing top 10 rows


Average Rating per Product (min 50 reviews)

In [0]:
from pyspark.sql.functions import avg

df.groupBy("asin") \
  .agg(
      count("*").alias("total_reviews"),
      avg("rating").alias("avg_rating")
  ) \
  .filter("total_reviews >= 50") \
  .orderBy("avg_rating", ascending=False) \
  .show(10)


+----------+-------------+------------------+
|      asin|total_reviews|        avg_rating|
+----------+-------------+------------------+
|B0002I1EW8|           54|               5.0|
|B00I3ELMX4|           57|               5.0|
|B08PFJSDFK|           59|               5.0|
|B07FYW4N7Q|           92| 4.989130434782608|
|B096PY225L|           62| 4.983870967741935|
|B07YK61YM6|           85| 4.976470588235294|
|B06X9QPNXF|           59| 4.966101694915254|
|B01A48JQ6Q|          102|  4.96078431372549|
|B07FKSDGML|           59|4.9491525423728815|
|B000SCLZNQ|           77|4.9480519480519485|
+----------+-------------+------------------+
only showing top 10 rows


Check FULL ROW DUPLICATES

In [0]:
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Duplicate rows:", total_rows - distinct_rows)

Total rows: 2607415
Distinct rows: 2578358
Duplicate rows: 29057


Duplicate reviews by same user for same product at same time

In [0]:
from pyspark.sql.functions import count

df.groupBy("user_id", "asin", "timestamp") \
  .count() \
  .filter("count > 1") \
  .show(10, truncate=False)


+----------------------------+----------+-------------+-----+
|user_id                     |asin      |timestamp    |count|
+----------------------------+----------+-------------+-----+
|AFHIS6C62UGA2SKV3M37TVTMEEJQ|B07MCZ9S8X|1560444001819|2    |
|AE7YXRXC6QAQCFRFNPSQ27KVPQXA|B07BX6SLZK|1590073157287|2    |
|AHW24URC4RFVEOKHFZXMVLNECLYQ|B09Q18Q3TF|1674304718805|2    |
|AGPV5HNR6FT7NS2WMURJXRN2PFTQ|B08DZJNGZ4|1613008637716|2    |
|AEKXGFKEHDSYJSOY4NLWL562L5TA|B00JH1K5GW|1478097296000|2    |
|AFLLXQEUFPYSLGPMDDRYKCIHXOXQ|B008J7VC8K|1643935458695|2    |
|AGMZKXCUDHUX6R2RJZAEUE3QV2KQ|B003L7VQMA|1474506729000|2    |
|AG3RQBTKLREULIDNZAT2W3UW7CFQ|B00ON9WQ1A|1437849956000|2    |
|AHJBP2NTU2FFU4POXKLOYMKEGP7A|B01JTDW9T8|1509587562295|2    |
|AHBZUJHBQTT2XLRRAJMAIQ6SDWYQ|B000LPSNRQ|1650490359540|2    |
+----------------------------+----------+-------------+-----+
only showing top 10 rows


Check Duplicate Reviews PER USER & PRODUCT

In [0]:
df.groupBy("user_id", "asin") \
  .count() \
  .filter("count > 1") \
  .orderBy("count", ascending=False) \
  .show(10)

+--------------------+----------+-----+
|             user_id|      asin|count|
+--------------------+----------+-----+
|AGALPU5ARZEK75CGK...|B009YC8JJE|   27|
|AEKEGPQQK4EI57DVH...|B0046IQ43O|    9|
|AHK5PAWXMBNYVC6Z4...|B00GYCDZF0|    9|
|AHF3UH4F4UFS6LGCI...|B071CZQDST|    9|
|AHI65SE5HVY22CDSF...|B000PR3JEM|    9|
|AG4SCNJCEYFBN43VX...|B019BNMVFI|    9|
|AHUS6HEGAICPDG3FV...|B0036ECH1M|    9|
|AHL67IRKKK44Z54JP...|B00VK4YXF8|    9|
|AGXGEFUONLS6OE6Q5...|B00VHKMK64|    9|
|AFZF2UH5WGHW4XG2O...|B000CZ0RLA|    9|
+--------------------+----------+-----+
only showing top 10 rows


% of Duplicate Rows

In [0]:
duplicate_pct = ((total_rows - distinct_rows) / total_rows) * 100
print(f"Duplicate percentage: {duplicate_pct:.4f}%")


Duplicate percentage: 1.1144%


Removing exacted duplicates

In [0]:
df_dedup = df.dropDuplicates()

In [0]:
df_dedup = df.dropDuplicates(["user_id", "asin", "timestamp"])

In [0]:
df_clean = df.dropDuplicates()

In [0]:
df_clean = df.dropDuplicates(["user_id", "asin", "timestamp"])

In [0]:
df_clean = df_clean.filter(df_clean.rating.isNotNull())
df_clean = df_clean.filter(df_clean.rating.between(1, 5))

In [0]:
from pyspark.sql.functions import from_unixtime, to_date

df_clean = df_clean.withColumn(
    "review_date",
    to_date(from_unixtime(df_clean.timestamp / 1000))
)

In [0]:
from pyspark.sql.functions import length

df_clean = df_clean.withColumn(
    "review_length",
    length("text")
)

In [0]:
from pyspark.sql.functions import when

df_clean = df_clean.withColumn(
    "sentiment",
    when(df_clean.rating >= 4, "Positive")
    .when(df_clean.rating == 3, "Neutral")
    .otherwise("Negative")
)

In [0]:
print("Before cleaning:", df.count())
print("After cleaning:", df_clean.count())

df_clean.printSchema()
df_clean.show(5, truncate=False)

Before cleaning: 2607415
After cleaning: 2578355
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- review_length: integer (nullable = true)
 |-- review_date: date (nullable = true)
 |-- sentiment: string (nullable = false)

+----------+------------+-----------+------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
output_path = "/Volumes/workspace/default/musical_instrument/cleaned_reviews_final_single_jsonl"
df_clean \
    .coalesce(1) \
    .write \
    .mode("error") \
    .json(output_path)